# Introduction to Large Language Models (LLMs)


## What is a Large Language Model?

Large Language Models (LLMs) are deep learning models trained on massive text corpora. They can understand and generate human-like text.

Popular LLMs include:

- GPT (OpenAI)
- Gemini (Google)
- Claude (Anthropic)
- LLaMA (Meta)
- DeepSeek

## Applications of LLMs

LLMs can be used in many tasks:

- Text summarization
- Translation
- Question answering
- Text generation
- Chatbots
- Sentiment analysis
- Code generation


## Limitations and Ethical Considerations

- May generate factually incorrect answers ("hallucination")
- Can reflect biases in training data
- Large environmental and computational cost
- Needs context-appropriate prompting

## Open-Source vs Commercial LLMs: Access and Use

LLMs come in two main forms:

### Commercial (Proprietary) APIs
- Closed weights
- Usually accessed via API
- Paid (may include free tier)
- Strong performance, frequent updates
- Examples: OpenAI GPT-4, Anthropic Claude, Google Gemini

###  Open-Source Models
- Weights are available for download
- Can be run locally or on your own infrastructure
- Community-supported, customizable
- Examples: Meta's LLaMA, Mistral, Falcon, DeepSeek


## Multimodal LLMs

Transformers can be used to process [text](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf) as well as [images](https://arxiv.org/abs/2010.11929) or other types of [data](https://arxiv.org/abs/2205.06175). 

Recently, LLMs are becoming multimodal and can process text and images in an integrated fashion.

LLMs are based on a neural network architecture called a **Transformer**, introduced in 2017 ([Attention is all you need, Vaswani et al. 2017](https://arxiv.org/pdf/1706.03762.pdf))

## Attention and transformers

In a self-attention layer, an input matrix $X$ ($n$ tokens of dimension $d$) are turned it into an output matrix $Z$ ($n$ components of dimension $d_v$) via three representational matrices of the input:

* queries Q
* keys K
* values V

$\Large {\rm Attention}(Q, K, V) = {\rm softmax}( Q \cdot K^T / \sqrt{d_k}) * V$

where $Q$, $K$ and $V$ are matrices representing linear transformations from the input matrix $x$ via learnable parameters $W^Q$, $W^K$ and $W^V$:

* $Q = X W^Q$
* $K = X W^K$
* $V = X W^V$

Note that 
* $X \in \mathbb{R}^{n \times d}$
* $Q \in \mathbb{R}^{n \times d_k}$
* $K \in \mathbb{R}^{n \times d_k}$
* $V \in \mathbb{R}^{n \times d_v}$
* $W^Q \in \mathbb{R}^{d \times d_k}$
* $W^K \in \mathbb{R}^{d \times d_k}$
* $W^V \in \mathbb{R}^{d_v \times d}$
* $Z \in \mathbb{R}^{n \times d_v}


**Self-attention:**

![self-attention](selfattention.png)


**Cross-attention:**

In cross-attention, an input matrix $X_1$ ($n$ tokens of dimension $d$) is turned it into an output matrix $Z$ ($n$ components of dimension $d_v$) contrasting with another input matrix $X_2$ via three representational matrices of the input, where we now have:

* $Q = X_1 W^Q$
* $K = X_2 W^K$
* $V = X_2 W^V$

Note that 
* $X_1 \in \mathbb{R}^{n \times d}$
* $X_2 \in \mathbb{R}^{m \times d}$
* $Q \in \mathbb{R}^{n \times d_k}$
* $K \in \mathbb{R}^{m \times d_k}$
* $V \in \mathbb{R}^{m \times d_v}$
* $W^Q \in \mathbb{R}^{d \times d_k}$
* $W^K \in \mathbb{R}^{d \times d_k}$
* $W^V \in \mathbb{R}^{d_v \times d}$
* $Z \in \mathbb{R}^{n \times d_v}

![cross-attention](cross-attention-summary.png)

A transformer uses several multi-head-attention layers to perform tasks such as translation, next token prediction, or even image classification.

![transformer](transformer.png)

In the case of the next token prediction, e.g. GPT, we only use the decoder part. 

During training, tokens are shifted one element to the right to be compared to the original values (the values to predict), properly masked to prevent the decoder from seeing future tokens. 

During inference, we predict one token at a time.

Example:

* Input tokens (shifted right):

```[CLS] The dog chased```

* Target tokens (what to predict):

```The dog chased the```

So the model learns:

From [CLS] → predict "The"

From "The" → predict "dog"

From "dog" → predict "chased"

From "chased" → predict "the"



### Masked self-atention

The attention formula is modified as follows:

$\Large {\rm Attention}(Q, K, V) = {\rm softmax}( Q \cdot K^T / \sqrt{d_k} + M) * V$

with M being a mask matrix, e.g.

|         | [CLS] | The | dog | chased |
| ------- | --- | --- | --- | --- |
| [CLS] | 0   | −∞  | −∞  | −∞  |
| The | 0   | 0   | −∞  | −∞  |
| dog | 0   | 0   | 0   | −∞  |
| chased | 0   | 0   | 0   | 0   |

so the softmax returns 0 for all future tokens in each row (they cannot pay attention to the future).


## Training LLMs

Large Language Models are usually trained in three phases:

### Unsupervised Pre-Training

We maximize the likelihood:

$\Large \sum_i \log P(u_i | u_{i-k}, ..., u_{i-1}; \theta)$

where we use a corpus of tokens $U=\lbrace{u_1, ..., u_n\rbrace}$

and where a Transformer Decoder Memory Compressed Attention ([T-DMCA](https://arxiv.org/abs/1801.10198)) model is used. It modified the transformer in three ways:

1. Decoder only: the encoder layer is removed to do next token prediction
2. Memory-compressed attention: the number of keys and values are reduced by doing a strided convolution.
3. Local attention: it divides the tokens into blocks of similar length and attention is performed in each block independently

![TDMCA](TDMCA.png)

### Supervised training


![openai](GPT.png)


The pretrained model can be modified and fine-tuned to solve some supervised tasks such as classification (sentiment analysis), entailment (logic analysis), similarity, and multiple choice in a supervised fashion ([Radford et al. 2018 (GPT)](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf)).



### Reinforcement Learning with Human Feedback (RLHF)

🔹 Supervised Fine-Tuning (SFT)

Human labelers provide example prompts and ideal responses.

The model is fine-tuned on this data to become better aligned with user expectations.

🔹 Reward Model (RM)

Collect human comparisons between two or more model outputs for the same prompt.

Train a separate reward model to predict which output is preferred.

E.g., "Output A is better than B" → RM learns a scoring function.

🔹 Reinforcement Learning (PPO)

Use Proximal Policy Optimization (PPO) (a reinforcement learning algorithm) to optimize the LLM so that its outputs maximize the reward model score.

The LLM becomes a policy that chooses tokens, and the RM guides it toward preferred behavior.

### Full pipeline

The full pipeline looks like this:

* Pretraining

Train a transformer on a massive dataset using next-token prediction (e.g., GPT-style language modeling).

* Supervised Fine-Tuning (SFT)

Fine-tune the model using human-written demonstrations of ideal behavior (e.g., answering politely, correcting mistakes).

* RLHF (Reinforcement Learning from Human Feedback)

Use reinforcement learning to further refine the model using human preference judgments.

##  Summarization with Hugging Face Transformers

We will use the `transformers` library to run a pre-trained summarization model: `facebook/bart-large-cnn`.


In [1]:
!pip install transformers -q

In [2]:
from transformers import pipeline

# Load a summarization pipeline
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

/home/fforster/anaconda3/lib/python3.8/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/fforster/anaconda3/lib/python3.8/site-packages/transformers/modeling_utils.py:415: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by th

In [3]:
article = """
ALeRCE is a real-time astronomical alert broker designed to process data from the Vera C. Rubin Observatory.
It classifies variable and transient phenomena such as supernovae and variable stars using machine learning algorithms.
It provides web interfaces and APIs to facilitate scientific exploration.
"""

summary = summarizer(article, max_length=50, min_length=10, do_sample=False)
print(summary[0]['summary_text'])

ALeRCE is a real-time astronomical alert broker. It classifies variable and transient phenomena such as supernovae and variable stars.
